# Code to Troubleshoot Rotom

In [43]:
import mujoco
import numpy as np

model = mujoco.MjModel.from_xml_path("robot.xml")   # change path if needed
data = mujoco.MjData(model)


In [44]:
print("Actuators:")
for i in range(model.nu):
    name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_ACTUATOR, i)
    print(i, name)

print("\nJoints:")
for i in range(model.njnt):
    name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_JOINT, i)
    print(i, name)

Actuators:
0 O
1 A
2 B
3 C

Joints:
0 O
1 A
2 B
3 C


In [45]:
aid = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_ACTUATOR, "O")
jid = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, "O")

qadr = model.jnt_qposadr[jid]
dadr = model.jnt_dofadr[jid]

print("actuator id for O =", aid)
print("joint id for O =", jid)
print("initial qpos =", data.qpos[qadr])
print("initial qvel =", data.qvel[dadr])

actuator id for O = 0
joint id for O = 0
initial qpos = 0.0
initial qvel = 0.0


In [46]:
data.ctrl[:] = 0.0

for _ in range(300):
    mujoco.mj_step(model, data)

data.ctrl[aid] = 0.5   # target angle in radians

for k in range(500):
    mujoco.mj_step(model, data)
    
    if k % 50 == 0:
        print(
            "step:", k,
            " qpos:", float(data.qpos[qadr]),
            " qvel:", float(data.qvel[dadr]),
            " force:", float(data.actuator_force[aid]),
            " ncon:", data.ncon
        )

for i in range(data.ncon):
    c = data.contact[i]

    g1 = c.geom1
    g2 = c.geom2

    b1 = model.geom_bodyid[g1]
    b2 = model.geom_bodyid[g2]

    geom1_name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_GEOM, g1)
    geom2_name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_GEOM, g2)

    body1_name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_BODY, b1)
    body2_name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_BODY, b2)

    print(f"contact {i}")
    print("  geom ids:", g1, g2)
    print("  geom names:", geom1_name, "<->", geom2_name)
    print("  body names:", body1_name, "<->", body2_name)
    print("  dist:", c.dist)

step: 0  qpos: 0.0031917820327366057  qvel: 1.59589100089067  force: 9.999999999259385  ncon: 0
step: 50  qpos: 0.3801761636710598  qvel: 1.8150985701616775  force: 0.6803195885113205  ncon: 0
step: 100  qpos: 0.4707826362134911  qvel: 0.38282983214911537  force: 0.22238312880329758  ncon: 0
step: 150  qpos: 0.48989235719270185  qvel: 0.08074195145586013  force: 0.1258116500515385  ncon: 0
step: 200  qpos: 0.49392275522058193  qvel: 0.017029151856624777  force: 0.10544389789681574  ncon: 0
step: 250  qpos: 0.49495523198380353  qvel: 0.008368765399338953  force: 0.0932088139946714  ncon: 0
step: 300  qpos: 0.4957251552459905  qvel: 0.007091000152337783  force: 0.07898407583449973  ncon: 0
step: 350  qpos: 0.49637757039998054  qvel: 0.006008791012119033  force: 0.0669297415013883  ncon: 0
step: 400  qpos: 0.4969304157302717  qvel: 0.005091745697807701  force: 0.05671510682359404  ncon: 0
step: 450  qpos: 0.4973988873125441  qvel: 0.004314657343823659  force: 0.0480594018565661  ncon: 0


In [47]:
model2 = mujoco.MjModel.from_xml_path("robot.xml")
data2 = mujoco.MjData(model2)

model2.opt.gravity[:] = 0.0

aid2 = mujoco.mj_name2id(model2, mujoco.mjtObj.mjOBJ_ACTUATOR, "O")
jid2 = mujoco.mj_name2id(model2, mujoco.mjtObj.mjOBJ_JOINT, "O")

qadr2 = model2.jnt_qposadr[jid2]
dadr2 = model2.jnt_dofadr[jid2]

data2.ctrl[:] = 0.0
for _ in range(300):
    mujoco.mj_step(model2, data2)

data2.ctrl[aid2] = 0.5

for k in range(500):
    mujoco.mj_step(model2, data2)
    
    if k % 50 == 0:
        print(
            "step:", k,
            " qpos:", float(data2.qpos[qadr2]),
            " qvel:", float(data2.qvel[dadr2]),
            " force:", float(data2.actuator_force[aid2]),
            " ncon:", data2.ncon
        )

step: 0  qpos: 0.003191664156831877  qvel: 1.5958320784159383  force: 10.0  ncon: 0
step: 50  qpos: 0.38017636638862917  qvel: 1.8151074321920444  force: 0.6803068170101838  ncon: 0
step: 100  qpos: 0.4707829364706007  qvel: 0.38282783850141294  force: 0.2223789291188929  ncon: 0
step: 150  qpos: 0.4898924785574589  qvel: 0.08074064742094163  force: 0.12581043771874967  ncon: 0
step: 200  qpos: 0.4939227936159445  qvel: 0.017028678496805952  force: 0.1054435735391781  ncon: 0
step: 250  qpos: 0.49495525342694213  qvel: 0.008368729618487832  force: 0.0932084180289241  ncon: 0
step: 300  qpos: 0.49572517341069794  qvel: 0.007090969982501765  force: 0.07898374025070787  ncon: 0
step: 350  qpos: 0.49637758578995095  qvel: 0.006008765465785795  force: 0.06692945716565468  ncon: 0
step: 400  qpos: 0.49693042877051996  qvel: 0.0050917240626118035  force: 0.05671486588995478  ncon: 0
step: 450  qpos: 0.4973988983626597  qvel: 0.004314639018210929  force: 0.048059197685768495  ncon: 0


In [48]:
model2 = mujoco.MjModel.from_xml_path("robot.xml")
data2 = mujoco.MjData(model2)

model2.opt.gravity[:] = 0.0

aid2 = mujoco.mj_name2id(model2, mujoco.mjtObj.mjOBJ_ACTUATOR, "O")
jid2 = mujoco.mj_name2id(model2, mujoco.mjtObj.mjOBJ_JOINT, "O")

qadr2 = model2.jnt_qposadr[jid2]
dadr2 = model2.jnt_dofadr[jid2]

data2.ctrl[:] = 0.0
for _ in range(300):
    mujoco.mj_step(model2, data2)

data2.ctrl[aid2] = 0.5

for k in range(500):
    mujoco.mj_step(model2, data2)
    
    if k % 50 == 0:
        print(
            "step:", k,
            " qpos:", float(data2.qpos[qadr2]),
            " qvel:", float(data2.qvel[dadr2]),
            " force:", float(data2.actuator_force[aid2]),
            " ncon:", data2.ncon
        )

step: 0  qpos: 0.003191664156831877  qvel: 1.5958320784159383  force: 10.0  ncon: 0
step: 50  qpos: 0.38017636638862917  qvel: 1.8151074321920444  force: 0.6803068170101838  ncon: 0
step: 100  qpos: 0.4707829364706007  qvel: 0.38282783850141294  force: 0.2223789291188929  ncon: 0
step: 150  qpos: 0.4898924785574589  qvel: 0.08074064742094163  force: 0.12581043771874967  ncon: 0
step: 200  qpos: 0.4939227936159445  qvel: 0.017028678496805952  force: 0.1054435735391781  ncon: 0
step: 250  qpos: 0.49495525342694213  qvel: 0.008368729618487832  force: 0.0932084180289241  ncon: 0
step: 300  qpos: 0.49572517341069794  qvel: 0.007090969982501765  force: 0.07898374025070787  ncon: 0
step: 350  qpos: 0.49637758578995095  qvel: 0.006008765465785795  force: 0.06692945716565468  ncon: 0
step: 400  qpos: 0.49693042877051996  qvel: 0.0050917240626118035  force: 0.05671486588995478  ncon: 0
step: 450  qpos: 0.4973988983626597  qvel: 0.004314639018210929  force: 0.048059197685768495  ncon: 0


In [49]:
model3 = mujoco.MjModel.from_xml_path("robot.xml")
data3 = mujoco.MjData(model3)

mujoco.mj_forward(model3, data3)

print("initial contacts:", data3.ncon)
for i in range(min(data3.ncon, 10)):
    c = data3.contact[i]
    print("contact", i, "geom1 =", c.geom1, "geom2 =", c.geom2, "dist =", c.dist)

initial contacts: 0
